<a href="https://colab.research.google.com/github/human720/Pembayaran-uas-/blob/main/Pembayaran_(uas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# PEMBAYARAN
# =========================
def pembayaran(user):
    riwayat_booking(user)
    kode = input("\nMasukkan kode booking yang akan dibayar: ").strip().upper()

    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("""
            SELECT id, status_booking, status_bayar
            FROM booking
            WHERE kode_booking = ? AND id_user = ?
        """, (kode, user["id"]))
        data = cur.fetchone()

    if not data:
        print("Kode booking tidak ditemukan.")
        return

    if data[1] == "DIBATALKAN":
        print("Booking sudah dibatalkan dan tidak dapat dibayar.")
        return

    if data[2] in ("LUNAS", "MENUNGGU VERIFIKASI"):
        print("Pembayaran untuk booking ini sudah diproses.")
        return

    print("\n1. Transfer Bank")
    print("2. Bayar di Tempat")
    pilihan = input("Pilih metode pembayaran: ").strip()

    if pilihan == "1":
        bank = input("Nama bank             : ").strip()
        referensi = input("Nomor referensi       : ").strip()
        if not bank or not referensi:
            print("Nama bank dan nomor referensi wajib diisi.")
            return
        metode = f"Transfer Bank - {bank}"
        status_bayar = "MENUNGGU VERIFIKASI"
        status_booking = "MENUNGGU PEMBAYARAN"
    elif pilihan == "2":
        metode = "Bayar di Tempat"
        referensi = ""
        status_bayar = "BELUM LUNAS"
        status_booking = "MENUNGGU PEMBAYARAN"
    else:
        print("Pilihan tidak tersedia.")
        return

    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO pembayaran
            (id_booking, metode, referensi, tanggal_bayar)
            VALUES (?, ?, ?, ?)
        """, (data[0], metode, referensi,
              datetime.now().strftime("%d-%m-%Y %H:%M")))
        cur.execute("""
            UPDATE booking
            SET status_bayar = ?, status_booking = ?
            WHERE id = ?
        """, (status_bayar, status_booking, data[0]))
        conn.commit()

    print(f"Pembayaran dicatat. Status: {status_bayar}")


def validasi_pembayaran():
    lihat_semua_booking()
    kode = input("\nKode booking yang diverifikasi: ").strip().upper()
    print("1. Terima pembayaran (LUNAS)")
    print("2. Tolak pembayaran (DITOLAK)")
    pilihan = input("Pilihan: ").strip()

    if pilihan == "1":
        status_bayar = "LUNAS"
        status_booking = "LUNAS"
    elif pilihan == "2":
        status_bayar = "DITOLAK"
        status_booking = "MENUNGGU PEMBAYARAN"
    else:
        print("Pilihan tidak tersedia.")
        return

    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("""
            UPDATE booking SET status_bayar = ?, status_booking = ?
            WHERE kode_booking = ? AND status_bayar = 'MENUNGGU VERIFIKASI'
        """, (status_bayar, status_booking, kode))
        conn.commit()
        if cur.rowcount == 0:
            print("Booking tidak ditemukan atau belum menunggu verifikasi.")
        else:
            print(f"Pembayaran berhasil diperbarui menjadi {status_bayar}.")